# 03 · A PINN with a closed-form Laplacian

Physics-informed neural networks need spatial derivatives of the network
output. The usual approach differentiates *through the network* with autograd,
which is the main accuracy/speed bottleneck for high-order PDEs.

`PINNHeat` instead gets `u_xx` from the **closed-form** second derivative of
the activation — no `autograd.grad` through the activation in the inner loop.
We solve the 1D heat equation `u_t = α·u_xx` with `u(x,0) = sin(πx)`, whose
analytic solution is `u(x,t) = e^{-απ²t} sin(πx)`.

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, ACCENT, GOOD, PRIMARY, WARM, INK
set_style()

from omnibias.torch.architectures import PINNHeat

torch.manual_seed(0)
ALPHA = 0.1
net = PINNHeat(hidden=64, base="softplus", alpha=ALPHA)
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
print(sum(p.numel() for p in net.parameters()), "parameters")

## Train

Loss = PDE residual on interior collocation points + initial-condition fit.
A few hundred Adam steps run in seconds on CPU.

In [ ]:
hist = []
for step in range(600):
    x = torch.rand(512)
    t = torch.rand(512) * 0.1
    _, residual = net(x, t)
    pde_loss = (residual**2).mean()

    x0 = torch.linspace(0.0, 1.0, 64)
    u0, _ = net(x0, torch.zeros_like(x0))
    ic_loss = ((u0 - torch.sin(torch.pi * x0)) ** 2).mean()

    loss = pde_loss + ic_loss
    opt.zero_grad(); loss.backward(); opt.step()
    hist.append((pde_loss.item(), ic_loss.item()))
    if step % 150 == 0:
        print(f"step {step:4d}  pde={pde_loss.item():.2e}  ic={ic_loss.item():.2e}")
print("done")

In [ ]:
import numpy as np
hist = np.array(hist)
fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.2))

axl.semilogy(hist[:, 0], color=PRIMARY, label="PDE residual")
axl.semilogy(hist[:, 1], color=ACCENT, label="initial condition")
axl.set_xlabel("step"); axl.set_ylabel("loss"); axl.set_title("Training loss"); axl.legend()

xg = torch.linspace(0, 1, 100)
for t_val, col in [(0.0, INK), (0.05, PRIMARY), (0.1, ACCENT)]:
    u, _ = net(xg, torch.full_like(xg, t_val))
    analytic = np.exp(-ALPHA * np.pi**2 * t_val) * np.sin(np.pi * xg.numpy())
    axr.plot(xg.numpy(), u.detach().numpy(), color=col, label=f"PINN t={t_val}")
    axr.plot(xg.numpy(), analytic, "--", color=col, lw=1.3)
axr.set_xlabel("x"); axr.set_title("Solution (solid) vs analytic (dashed)"); axr.legend()
plt.tight_layout(); plt.show()

## Takeaway

The PINN matches the analytic decay `e^{-απ²t} sin(πx)` using a closed-form
`u_xx`. For higher-order PDEs (biharmonic, Navier–Stokes via the
streamfunction cage in `omnibias-pinn`) this is where the accuracy and speed
wins compound.

Next: **[04 · Schrödinger / QPINN](04_qpinn_schrodinger.ipynb)**.